# Compact surfaces from fundamental domains

A `Circle`, a `Torus2D` and a `Sphere` are the domains you can write down by
name. `FundamentalDomain` is how you get the rest: give it a polygon and a rule
for gluing the sides in pairs, and it presents the surface that gluing produces.
Every closed surface arises this way, so the six constructors below reach the
whole classification -- orientable or not, flat or curved.

What the simulator needs from a domain is a `distance`, and on a glued surface
that is the *quotient* distance: the shortest hop from one point to any image of
the other under the group generated by the pairings. That single change is what
makes self-excitation wrap around a handle.

This notebook executes on every documentation build. It is budgeted to finish in
well under a minute, which is why the one simulation below runs on a flat
surface -- see the last section for what the hyperbolic ones cost.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import hawkes_package as hp

## Six surfaces, by name

Each constructor returns a polygon with its pairings already set, and a
`topology` that says which surface came out. The Euler characteristic `chi` is
the invariant that identifies it: `chi = 2 - 2g` for an orientable surface of
genus `g`, and `chi = 2 - k` for `k` crosscaps.

`nodes_per_axis` is the coarsest tensor quadrature rule that measures this
polygon's area correctly, and it is the number to watch -- it is what the cost of
working on the surface scales with.

In [ ]:
domains = {
    "rectangle(2pi, 2pi)": hp.FundamentalDomain.rectangle(2 * np.pi, 2 * np.pi),
    "hexagon(1.0)": hp.FundamentalDomain.hexagon(1.0),
    "klein_bottle()": hp.FundamentalDomain.klein_bottle(2 * np.pi, 2 * np.pi),
    "projective_plane()": hp.FundamentalDomain.projective_plane(),
    "crosscaps(3)": hp.FundamentalDomain.crosscaps(3),
    "genus(2)": hp.FundamentalDomain.genus(2),
    "genus(3)": hp.FundamentalDomain.genus(3),
}

header = f"{'constructor':22s} {'sides':>5s} {'chi':>4s} {'orient':>7s} {'area':>9s} {'nodes':>6s}"
print(header, " surface")
for name, domain in domains.items():
    topology = domain.topology
    print(
        f"{name:22s} {len(domain.vertices):5d} {topology.euler_characteristic:4d} "
        f"{topology.orientable!s:>7s} {domain.volume:9.4f} "
        f"{domain.nodes_per_axis:6d}  {topology.name}"
    )

## The gluing, and what "nearby" means

The Klein bottle is a rectangle with one pair of opposite sides glued straight
across and the other pair glued with a flip. `orbit` shows the consequence: a
point in the polygon stands for a whole family of points in the plane, and the
surface cannot tell them apart.

So two points sitting at opposite edges of the polygon are *neighbours* on the
surface, however far apart they look on the page. That is the whole content of
using a quotient distance, and it is why a kernel of the distance excites across
the boundary rather than stopping at it.

In [ ]:
bottle = domains["klein_bottle()"]
width = height = 2 * np.pi
here, across = np.array([2.8, 1.0]), np.array([-2.8, 1.0])
images = np.array(bottle.orbit(here, n_images=1))
nearest = images[int(np.argmin([np.linalg.norm(image - across) for image in images]))]

fig, ax = plt.subplots(figsize=(5.6, 5.6))
ax.add_patch(
    plt.Rectangle((-width / 2, -height / 2), width, height, fill=False, ec="k", lw=1.5, zorder=2)
)
ax.plot(
    [across[0], here[0]],
    [across[1], here[1]],
    ":",
    c="grey",
    lw=1.4,
    label="straight across the page",
)
ax.plot(
    [across[0], nearest[0]],
    [across[1], nearest[1]],
    "-",
    c="tab:red",
    lw=2.2,
    label="the route the surface takes",
)
ax.scatter(
    images[1:, 0], images[1:, 1], s=42, c="tab:orange", zorder=3, label="images of the blue point"
)
ax.scatter(*here, s=110, c="tab:blue", zorder=4, label="a point in the polygon")
ax.scatter(*across, s=110, c="tab:green", marker="s", zorder=4, label="a point at the far edge")
ax.set_xlim(-width, width)
ax.set_ylim(-height, height)
ax.set_aspect("equal")
ax.set_title("the Klein bottle's polygon, and one point's images")
ax.legend(fontsize=7.5, loc="lower center", ncol=2, framealpha=0.9)
fig.tight_layout()

In [ ]:
euclidean = float(np.linalg.norm(here - across))
on_surface = bottle.distance(here, across)
hops = [float(np.linalg.norm(image - across)) for image in images]

print(f"straight across the page: {euclidean:.4f}")
print(f"on the surface:           {on_surface:.4f}")
print(f"shortest hop to an image: {min(hops):.4f}")
print(f"the gluing brings them {euclidean / on_surface:.1f}x closer")

## Which geometry, and why there is no choice

The polygon cannot be drawn in whichever plane you like. Gauss-Bonnet ties the
total curvature to the topology, so once `chi` is fixed the geometry is too:
positive `chi` is spherical, zero is flat, negative is hyperbolic. For the
hyperbolic ones the area is not a free parameter either -- it is exactly
`-2 * pi * chi`.

This is why `genus(2)` returns a domain in the hyperbolic plane and not a
strangely-shaped Euclidean octagon: no Euclidean octagon has the angle sums the
gluing requires.

In [ ]:
print(f"{'surface':32s} {'chi':>4s} {'area':>9s} {'-2*pi*chi':>10s}  geometry")
for name, domain in domains.items():
    chi = domain.topology.euler_characteristic
    geometry = "spherical" if chi > 0 else ("flat" if chi == 0 else "hyperbolic")
    predicted = f"{-2 * np.pi * chi:10.4f}" if chi < 0 else f"{'-':>10s}"
    print(f"{name:32s} {chi:4d} {domain.volume:9.4f} {predicted}  {geometry}")

## A domain that is not its bounding box

A hexagon does not fill the rectangle around it, and that matters for more than
tidiness: the quadrature the simulator integrates against is a *tensor* rule on
the bounding box, so some of its nodes fall outside the domain. `contains` masks
them and `volume_element` reweights the survivors, which is what lets a domain be
a proper subset of its box at all.

This does not weaken the thinning bound. That argument only ever needed the bound
and the acceptance test to share one node set with strictly positive weights --
never that the nodes fill a box.

In [ ]:
hexagon = domains["hexagon(1.0)"]
box = hexagon.bounds
box_area = float(np.prod(np.diff(box, axis=1)))

count = 24
grid = np.array(
    [
        [x, y]
        for y in np.linspace(box[1, 0], box[1, 1], count)
        for x in np.linspace(box[0, 0], box[0, 1], count)
    ]
)
inside = np.array([hexagon.contains(point) for point in grid])

print(
    f"hexagon area {hexagon.volume:.4f}, bounding box {box_area:.4f} "
    f"-> {100 * hexagon.volume / box_area:.1f}% of the box"
)
print(f"a {count}x{count} tensor rule keeps {inside.sum()} of {inside.size} nodes")

In [ ]:
loop = np.vstack([hexagon.vertices, hexagon.vertices[:1]])

fig, ax = plt.subplots(figsize=(5.0, 5.0))
ax.plot(loop[:, 0], loop[:, 1], "k-", lw=1.5, zorder=3)
ax.scatter(grid[inside, 0], grid[inside, 1], s=8, c="tab:blue", label="kept")
ax.scatter(grid[~inside, 0], grid[~inside, 1], s=8, c="tab:red", alpha=0.45, label="masked out")
ax.set_aspect("equal")
ax.set_title("the tensor rule, masked by contains()")
ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()

## Simulating on the hexagonal torus

The same `SpatioTemporalHawkesProcess` as anywhere else -- the domain is the only
thing that changed. Six events, because each one costs a quadrature sweep over
the masked grid plus a Metropolis chain for its location, and this notebook runs
on every docs build.

The spatial kernel reaches 0.6, which is comparable to the hexagon's own size, so
the excitation genuinely wraps: an event near one side raises the intensity near
the side it is glued to.

In [ ]:
def temporal(lag):
    return 0.6 * np.exp(-2.0 * lag)


def spatial(distance):
    return np.where(distance < 0.6, np.cos(distance / 0.6 * np.pi / 2) ** 2, 0.0)


process = hp.SpatioTemporalHawkesProcess(
    base=lambda x: 0.4,
    spatial=spatial,
    temporal=temporal,
    domain=hexagon,
    rng=11,
)
process.simulate(6)
events = process.events

print(f"{events.shape[1]} events, last at t = {events[0, -1]:.2f}")
print(
    "every event on the domain:",
    all(hexagon.contains(events[1:, i]) for i in range(events.shape[1])),
)

In [ ]:
fig, ax = plt.subplots(figsize=(5.0, 5.0))
ax.plot(loop[:, 0], loop[:, 1], "k-", lw=1.5)
dots = ax.scatter(events[1], events[2], c=events[0], cmap="viridis", s=80, zorder=3)
fig.colorbar(dots, ax=ax, label="time")
ax.set_aspect("equal")
ax.set_title("events on the hexagonal torus")
fig.tight_layout()

## Where it stops, and why

Twelve sides -- genus 3, or six crosscaps. The limit is the search, not the
geometry, and the table below is why.

A certified `distance` has to enumerate every deck-group element within a radius
that scales with the polygon, and the element count grows like `exp(R)`. Two
things grow with it: `nodes_per_axis`, and so the quadrature grid as its square;
and the work each individual `distance` call does. A genus-3 surface needs 64
times the quadrature nodes of a flat one, and pays more per node.

Deliberately no timings here -- they would differ on every machine that builds
these docs. The structural numbers are the honest ones, and they are enough.

In [ ]:
print(f"{'surface':22s} {'sides':>5s} {'nodes':>6s} {'grid':>8s} {'vs flat':>8s}")
flat = domains["hexagon(1.0)"].nodes_per_axis ** 2
for name, domain in domains.items():
    grid_size = domain.nodes_per_axis**2
    print(
        f"{name:22s} {len(domain.vertices):5d} {domain.nodes_per_axis:6d} "
        f"{grid_size:8d} {grid_size / flat:7.0f}x"
    )

In [ ]:
# The refusal is at construction, with the reason, rather than a run that never
# finishes.
try:
    hp.FundamentalDomain.genus(4)
except ValueError as exc:
    print(f"Caught expected error:\n  {exc}")

## Where to go next

- [Spatio-temporal Hawkes processes](spatio_temporal.ipynb) -- the intensity
  field, and the same process on a circle, a torus and a sphere.
- [Theory](../theory.md#fundamental-domains) -- what makes a presentation a
  surface, how orientability falls out of the pairings, and the argument for
  truncating an infinite group.
- [Quickstart](../quickstart.md#your-own-domain) -- writing a domain of your own
  against the `SpatialDomain` contract.